# 06 · Marker Gene Analysis (Eexploratory)

**Goal**: Identify marker genes enriched in each Leiden cluster using the Wilcoxon rank-sum test to explore their biological characteristics.

**Input**: results/adata/05_spatial.h5ad

**Output**: results/adata/06_markers.h5ad, marker gene tables, and figures

**Note**: This analysis is exploratory. The 56 µm spatial bins may contain multiple cells, and QC filtering was not applied. Marker genes should not be interpreted as definitive cell-type identities.

## 1. Imports and paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt



def setup_marker_paths():
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "results/adata/05_spatial.h5ad").exists()), None)
    if root is None:
        raise FileNotFoundError("Run Step 05 first: results/adata/05_spatial.h5ad not found")
    INPUT = root / "results/adata/05_spatial.h5ad"
    OUTPUT = root / "results/adata/06_markers.h5ad"
    FIG = root / "results/figures"
    TAB = root / "results/tables"
    for p in (OUTPUT.parent, FIG, TAB):
        p.mkdir(parents=True, exist_ok=True)
    SEED = 42
    np.random.seed(SEED)
    print("Python:", sys.executable)
    print("Scanpy:", version("scanpy"), "AnnData:", version("anndata"))
    print("Project root:", root)
    return root, INPUT, OUTPUT, FIG, TAB, SEED

ROOT, INPUT, OUTPUT, FIG, TAB, SEED = setup_marker_paths()

## 2. Load and validate spatial analysis results

Load the AnnData file from Step 05 and verify that Leiden clusters, spatial neighbor graphs, and raw counts are preserved. Summarize the number of bins in each cluster and save the results as a CSV file.

In [ ]:
def load_and_validate_spatial(input_path, table_dir):
    adata = sc.read_h5ad(input_path)
    assert "leiden" in adata.obs and "spatial" in adata.obsm
    assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
    assert "spatial_connectivities" in adata.obsp and "connectivities" in adata.obsp
    assert "log1p" in adata.uns, "Expected log1p-normalized X from Step 03"
    adata.obs["leiden"] = adata.obs["leiden"].astype("category")
    cluster_sizes = adata.obs["leiden"].value_counts().sort_index()
    cluster_sizes.rename_axis("leiden").reset_index(name="n_bins").to_csv(
        table_dir / "06_cluster_sizes.csv", index=False)
    print("AnnData:", adata.shape, "Clusters:", adata.obs["leiden"].nunique())
    print("Smallest cluster:", cluster_sizes.min(), "bins")
    print("Using log1p-normalized expression in adata.X (not raw counts)")
    return adata, cluster_sizes

adata, cluster_sizes = load_and_validate_spatial(INPUT, TAB)

## 3. Differential expression across spatial-bin clusters

Identify marker genes for each Leiden cluster by comparing its gene expression with all other bins using the Wilcoxon rank-sum test. Apply multiple-testing correction and rank genes based on differential expression. Results are exploratory because spatial dependence is not accounted for.

In [ ]:
def rank_cluster_markers(adata):
    KEY = "rank_genes_leiden_wilcoxon"
    sc.tl.rank_genes_groups(
        adata, groupby="leiden", method="wilcoxon", reference="rest",
        use_raw=False, n_genes=adata.n_vars, pts=True,
        key_added=KEY, tie_correct=False,
    )
    print("PASS — Marker ranking calculated for", adata.obs["leiden"].nunique(), "clusters")
    return KEY

KEY = rank_cluster_markers(adata)

## 4. Export complete results and filtered marker shortlist

Export differential expression results for all Leiden clusters. Filter candidate marker genes using adjusted p-values (less than 0.05) and log2 fold changes (at least 0.5), then select the top 10 markers per cluster. Save the complete results, shortlisted markers, and cluster summaries as CSV files.

In [ ]:
def export_marker_tables(adata, cluster_sizes, key, table_dir):
    frames = []
    for cluster in adata.obs["leiden"].cat.categories:
        df = sc.get.rank_genes_groups_df(adata, group=str(cluster), key=key)
        df.insert(0, "leiden", str(cluster))
        frames.append(df)
    markers = pd.concat(frames, ignore_index=True)
    markers.to_csv(table_dir / "06_all_marker_statistics.csv", index=False)

    # Screening criteria are adjustable, not biological validation.
    MIN_LOGFC = 0.5
    MAX_ADJ_P = 0.05
    TOP_N = 10
    filtered = markers.loc[
        (markers["logfoldchanges"] >= MIN_LOGFC)
        & (markers["pvals_adj"] < MAX_ADJ_P)
    ].copy()
    filtered = filtered.sort_values(["leiden", "scores"], ascending=[True, False])
    top = filtered.groupby("leiden", sort=False).head(TOP_N).copy()
    top.to_csv(table_dir / "06_top10_markers_per_cluster.csv", index=False)
    summary = pd.DataFrame({
        "leiden": cluster_sizes.index.astype(str),
        "n_bins": cluster_sizes.to_numpy(),
        "n_markers_padj_lt_0_05_logfc_ge_0_5": [
            int((filtered["leiden"] == str(c)).sum()) for c in cluster_sizes.index
        ],
    })
    summary.to_csv(table_dir / "06_marker_cluster_summary.csv", index=False)
    print("Total gene–cluster comparisons:", len(markers))
    print("Shortlisted markers:", len(top))
    print("Clusters with >=1 shortlisted marker:", top["leiden"].nunique())
    display(top.head(20))
    return markers, filtered, top, MIN_LOGFC, MAX_ADJ_P, TOP_N

markers, filtered, top, MIN_LOGFC, MAX_ADJ_P, TOP_N = export_marker_tables(adata, cluster_sizes, KEY, TAB)

## 5. Visualize cluster sizes and representative marker expression

Visualize the number of spatial bins in each Leiden cluster. Select representative marker genes and display their expression patterns across clusters using dot plots and heatmaps.

In [ ]:
def plot_marker_summaries(adata, cluster_sizes, top, figure_dir):
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(cluster_sizes.index.astype(str), cluster_sizes.values)
    ax.set(xlabel="Leiden cluster", ylabel="Number of spatial bins", title="Spatial-bin cluster sizes")
    ax.tick_params(axis="x", rotation=90)
    fig.tight_layout()
    fig.savefig(figure_dir / "06_cluster_sizes.png", dpi=170)
    plt.show()

    # Select one leading marker per cluster, then deduplicate gene names.
    representatives = (top.groupby("leiden", sort=False).head(1)["names"].drop_duplicates().tolist())
    if representatives:
        # Cap to keep plots readable; full marker table is always exported.
        selected = representatives[:30]
        sc.pl.dotplot(adata, var_names=selected, groupby="leiden", use_raw=False,
                      standard_scale="var", show=False)
        plt.savefig(figure_dir / "06_marker_dotplot.png", dpi=170, bbox_inches="tight")
        plt.show()
        sc.pl.matrixplot(adata, var_names=selected, groupby="leiden", use_raw=False,
                         standard_scale="var", cmap="viridis", show=False)
        plt.savefig(figure_dir / "06_marker_heatmap.png", dpi=170, bbox_inches="tight")
        plt.show()
    else:
        print("No markers pass screening thresholds; plots skipped. Inspect full table.")

plot_marker_summaries(adata, cluster_sizes, top, FIG)

## 6. Spatial expression maps for candidate genes

Visualize the spatial expression patterns of selected candidate genes using full-resolution pixel coordinates. Expression levels are displayed without a tissue-image overlay because image registration has not been verified.

In [ ]:
def plot_candidate_spatial_expression(adata, figure_dir):
    CANDIDATES = ["COL1A1", "COL2A1", "ACAN", "SPP1", "BGLAP", "DCN", "LUM", "PTPRC", "PECAM1"]
    genes = [g for g in CANDIDATES if g in adata.var_names][:6]
    if genes:
        xy = np.asarray(adata.obsm["spatial"])
        fig, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
        for ax, gene in zip(axes.flat, genes):
            x = adata[:, gene].X
            expr = x.toarray().ravel() if sparse.issparse(x) else np.asarray(x).ravel()
            points = ax.scatter(xy[:, 0], xy[:, 1], c=expr, s=3, cmap="magma", linewidths=0)
            ax.set(title=gene, xlabel="Fullres x (pixels)", ylabel="Fullres y (pixels)")
            ax.invert_yaxis()
            ax.set_aspect("equal", adjustable="box")
            fig.colorbar(points, ax=ax, fraction=0.045, pad=0.02)
        for ax in axes.flat[len(genes):]: ax.axis("off")
        fig.suptitle("Candidate gene expression — spatial coordinates only")
        fig.tight_layout()
        fig.savefig(figure_dir / "06_candidate_gene_spatial.png", dpi=170)
        plt.show()
    else:
        print("No candidate genes found; spatial expression map skipped")

plot_candidate_spatial_expression(adata, FIG)

## 7. Save and verify marker analysis results

Save the marker analysis results and processing settings in an AnnData file. Reload the saved file to verify that marker statistics, Leiden clusters, spatial coordinates, neighbor graphs, and raw counts are preserved.

In [ ]:
def save_and_verify_markers(adata, output_path, table_dir, root, key, min_logfc, max_adj_p, top_n, markers):
    adata.uns["step06_settings"] = {
        "method": "wilcoxon", "reference": "rest", "groupby": "leiden",
        "marker_key": key, "min_logfoldchange": float(min_logfc),
        "max_adjusted_p": float(max_adj_p), "top_n": int(top_n),
        "exploratory_only": True, "qc_filters_applied": False,
        "image_registration_verified": False,
    }
    adata.write_h5ad(output_path, compression="gzip")
    check = sc.read_h5ad(output_path)
    assert check.shape == adata.shape
    assert check.obs["leiden"].equals(adata.obs["leiden"])
    assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
    assert "spatial_connectivities" in check.obsp and "connectivities" in check.obsp
    assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
    assert key in check.uns
    assert (table_dir / "06_all_marker_statistics.csv").exists()
    print("PASS — 06_marker_analysis completed")
    print("AnnData:", check.shape)
    print("Leiden clusters:", check.obs["leiden"].nunique())
    print("All marker comparisons:", len(markers))
    print("Output:", output_path.relative_to(root))

save_and_verify_markers(adata, OUTPUT, TAB, ROOT, KEY, MIN_LOGFC, MAX_ADJ_P, TOP_N, markers)

## Next: 07. Biological Annotation

In `07_biological_annotation.ipynb`, we will use candidate marker genes to explore the biological characteristics of each Leiden cluster.

**Input:** `results/adata/06_markers.h5ad`

**Note:** Biological annotations are exploratory and do not represent validated cell identities.